# 📊 YouTube Yorumları ile Türkçe Duygu Analizi (LSTM)
YouTube API’den yorumları çekip, LSTM ile duygu analizi yapıyoruz.

In [3]:
!pip install -q tensorflow pandas scikit-learn google-api-python-client

In [4]:
from googleapiclient.discovery import build
import pandas as pd

# API anahtarınızı ve video ID'nizi girin
api_key = 'REDACTED_GOOGLE_API_KEY'
video_id = 'yP2weCj6yaM'

youtube = build('youtube', 'v3', developerKey=api_key)
comments = []
next_page_token = None

while True:
    response = youtube.commentThreads().list(
        part='snippet',
        videoId=video_id,
        maxResults=100,
        pageToken=next_page_token
    ).execute()

    for item in response['items']:
        text = item['snippet']['topLevelComment']['snippet']['textDisplay']
        comments.append(text)

    next_page_token = response.get('nextPageToken')
    if not next_page_token:
        break

print(f'Toplam çekilen yorum sayısı: {len(comments)}')
df = pd.DataFrame({'yorum': comments})
df.head()

Toplam çekilen yorum sayısı: 904


,yorum
0,Arda&#39;nın sıradan bir topçu olduğu açık.
1,Helal olsun bizim çocuklar takımlarını 10ar ki...
2,O değilde sergenin denizliye attığı frikik gol...
3,Kenan bence vini gibi oynadi hatta indan da iy...
4,Bal Verde güzel şutlar vurdu


In [5]:
import re
def temizle(metin):
    metin = metin.lower()
    metin = re.sub(r"http\S+", "", metin)
    metin = re.sub(r"[^a-zçğıöşü\s]", "", metin)
    metin = re.sub(r"\s+", " ", metin).strip()
    return metin

df['yorum_temiz'] = df['yorum'].apply(temizle)
df[['yorum', 'yorum_temiz']].head()

,yorum,yorum_temiz
0,Arda&#39;nın sıradan bir topçu olduğu açık.,ardanın sıradan bir topçu olduğu açık
1,Helal olsun bizim çocuklar takımlarını 10ar ki...,helal olsun bizim çocuklar takımlarını ar kişi...
2,O değilde sergenin denizliye attığı frikik gol...,o değilde sergenin denizliye attığı frikik gol...
3,Kenan bence vini gibi oynadi hatta indan da iy...,kenan bence vini gibi oynadi hatta indan da iy...
4,Bal Verde güzel şutlar vurdu,bal verde güzel şutlar vurdu


In [6]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Etiketleme: manuel veya hazır veriyle
# Örneğin küçük test için basit pozitif/negatif etiketleme:
labels = [1 if 'harika' in y or 'mükemmel' in y else 0 for y in df['yorum_temiz']]

tokenizer = Tokenizer(num_words=1000)
tokenizer.fit_on_texts(df['yorum_temiz'])

X = tokenizer.texts_to_sequences(df['yorum_temiz'])
X_pad = pad_sequences(X, maxlen=20)
y = labels

In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential()
model.add(Embedding(input_dim=1000, output_dim=64, input_length=20))
model.add(LSTM(64, dropout=0.2))
model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [8]:
model.fit(X_pad, y, epochs=10, batch_size=32, validation_split=0.2)

ValueError: Argument `validation_split` is only supported for tensors or NumPy arrays.Found incompatible type in the input: [<class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>, <clas

In [ ]:
def predict_yorum(yeni_yorum):
    temiz = temizle(yeni_yorum)
    seq = tokenizer.texts_to_sequences([temiz])
    pad = pad_sequences(seq, maxlen=20)
    tahmin = model.predict(pad)[0][0]
    return f"'{yeni_yorum}' → {'Pozitif' if tahmin > 0.5 else 'Negatif'} ({tahmin:.2f})"

print(predict_yorum("Bu video çok güzel olmuş"))